# Day 3 OOP Notebook

This notebook is a practical healthcare example of object-oriented programming.

What you will learn in this file:

1. What a class and object are.
2. How inheritance lets us reuse code.
3. How polymorphism lets the same method behave differently.
4. How type hints make code clearer.
5. How dataclasses reduce code writing.
6. How factory and singleton patterns help in real software.
7. How a typed data pipeline works with `load()`, `transform()`, and `export()`.

Why this matters:

- In real projects, we do not write everything in one long script.
- We organize code into small reusable classes.
- This makes the code easier to read, test, and maintain.

Important note:

- A class is a blueprint.
- An object is one real thing made from that blueprint.
- A method is a function inside a class.
- `self` means “this current object”.


In [ ]:
# ------------------------------------------------------------------
# Cell 1: Imports and dataset loading
# ------------------------------------------------------------------
# pandas is the main library we use for data tables in Python.
# It lets us read CSV files and inspect rows like a spreadsheet.


# pathlib.Path helps us build safe file paths.
# This is important because the notebook may run from two different folders.
import pandas as pd
from pathlib import Path

# Try the workspace-style path first.
# This works when we launch the notebook from the AI_learning root folder.
data_path = Path('Day3_OOP_DataPipeline/data/healthcare.csv')

# If that path does not exist, try the shorter path.
# This works when the notebook is opened inside the Day3_OOP_DataPipeline folder.
if not data_path.exists():
    data_path = Path('data/healthcare.csv')

# Read the CSV data into a DataFrame object.
# A DataFrame is a table with rows and columns, similar to Excel.
df = pd.read_csv(data_path)

# Show the first 5 rows of the dataset.
# This helps us confirm the data loaded correctly before applying OOP ideas.
df.head()

ImportError: C extension: pandas.compat not built. If you want to import pandas from the source directory, you may need to run 'python -m pip install -ve . --no-build-isolation -Ceditable-verbose=true' to build the C extensions first.

## 1. Basic Class Example

This section is the first real OOP step.

- `PatientRecord` is the class blueprint.
- `sample` is one object created from that blueprint.
- `__init__()` sets the starting values of the object.
- `cost_per_visit()` computes average cost.
- `is_high_cost()` returns True or False.

Why this matters:
A class groups related data and behavior together. Instead of keeping patient values in separate variables, we place them in one object.


In [ ]:
# ------------------------------------------------------------------
# Cell 2: Basic class example
# ------------------------------------------------------------------
# A class is a blueprint for creating objects.
# Here, PatientRecord is the blueprint for one person in the healthcare dataset.
class PatientRecord:
    # __init__ runs automatically whenever we create a new object.
    # It stores the initial values that belong to this patient.
    def __init__(self, patient_id: int, age: int, total_cost: float, visits: int):
        # Save the patient ID in the object.
        self.patient_id = patient_id
        # Save the age in the object.
        self.age = age
        # Save the total cost in the object.
        self.total_cost = total_cost
        # Save the number of visits in the object.
        self.visits = visits

    # This method calculates the average cost spent per visit.
    def cost_per_visit(self) -> float:
        # Total cost divided by visit count gives average cost per visit.
        return self.total_cost / self.visits

    # This method checks whether the patient is high cost.
    def is_high_cost(self) -> bool:
        # If the patient spent more than 2000, we label them high cost.
        return self.total_cost > 2000

# Create one actual object from the class blueprint.
# This object is called sample and stores one patient record.
sample = PatientRecord(1001, 45, 2200.50, 6)

# Print the result of the calculation.
print('Cost per visit:', round(sample.cost_per_visit(), 2))

# Print whether this patient is considered high cost.
print('High cost:', sample.is_high_cost())

Cost per visit: 366.75
High cost: True


## 2. Inheritance and Polymorphism

This section explains the most important OOP idea after classes.

- Inheritance means one class takes behavior from another parent class.
- Polymorphism means different classes can use the same method name but behave differently.
- Here, `BasePatient` gives the basic patient description.
- `HealthcarePatient` adds condition and department.
- `RiskPatient` changes the description to show a high-risk patient.

In simple words:
The same `describe()` method name is used in different classes, but each class gives it its own behavior.


In [ ]:
# ------------------------------------------------------------------
# Cell 3: Inheritance and polymorphism
# ------------------------------------------------------------------
# BasePatient is the parent class.
# It defines what all patient objects have in common.
class BasePatient:
    def __init__(self, patient_id: int, age: int):
        # Store shared attributes that every patient has.
        self.patient_id = patient_id
        self.age = age

    # This method gives a common description for all patients.
    def describe(self) -> str:
        return f'Patient {self.patient_id} is {self.age} years old.'


# HealthcarePatient inherits from BasePatient.
# It adds extra healthcare-specific information.
class HealthcarePatient(BasePatient):
    def __init__(self, patient_id: int, age: int, condition: str, department: str):
        # super() calls the parent class constructor.
        super().__init__(patient_id, age)
        # Add new attributes for this subclass.
        self.condition = condition
        self.department = department

    # This version of describe() overrides the parent method.
    # The method name is the same, but the behavior is different.
    def describe(self) -> str:
        return f'{super().describe()} Condition: {self.condition}, Department: {self.department}'


# RiskPatient inherits from HealthcarePatient.
# It shows a more specific patient type.
class RiskPatient(HealthcarePatient):
    # This overrides describe() again to add high-risk wording.
    def describe(self) -> str:
        return 'High-risk patient: ' + super().describe()


# Create two different objects to compare them.
patients = [
    HealthcarePatient(1001, 45, 'Diabetes', 'Endocrinology'),
    RiskPatient(1008, 63, 'Diabetes', 'Endocrinology'),
]

# Loop through each object and print its description.
# Even though we call the same method name, Python uses the correct version.
for patient in patients:
    print(patient.describe())

Patient 1001 is 45 years old. Condition: Diabetes, Department: Endocrinology
High-risk patient: Patient 1008 is 63 years old. Condition: Diabetes, Department: Endocrinology


## 3. Type Hints and Dataclasses

This section introduces two Python tools that make OOP cleaner.

- Type hints such as `patient_id: int` tell us what each field should contain.
- Dataclasses automatically create the constructor and simple printing behavior.
- This makes code shorter, cleaner, and easier to maintain.

Use this example when you want to model simple records like patient summaries or invoices.


In [ ]:
# ------------------------------------------------------------------
# Cell 4: Type hints and dataclasses
# ------------------------------------------------------------------
# dataclass automatically creates a simple constructor and readable string output.
# It is useful for record-style data objects.
from dataclasses import dataclass


@dataclass
class PatientSummary:
    # These type hints tell us what type each field must store.
    patient_id: int
    age: int
    condition: str
    total_cost: float
    visits: int

    # This method calculates cost per visit using the record data.
    def cost_per_visit(self) -> float:
        return self.total_cost / self.visits


# Create one dataclass object with healthcare values.
summary = PatientSummary(1004, 41, 'Diabetes', 2400.25, 7)

# Print the object itself.
# dataclass gives us a clean and readable representation.
print(summary)

# Print the calculated average cost per visit.
print('Cost per visit:', round(summary.cost_per_visit(), 2))

## 4. Factory Pattern

This section introduces the factory pattern.

- The factory decides which object should be created.
- It checks the incoming record and chooses the most suitable class.
- This prevents repeated manual logic in the main program.

Why this is useful:
In real applications, you often receive mixed data. A factory hides that decision-making logic from the rest of the code.


In [ ]:
# ------------------------------------------------------------------
# Cell 5: Factory pattern
# ------------------------------------------------------------------
# A factory pattern is a way to create the correct object automatically.
# Instead of writing many if/else blocks in the main code, the factory handles it.
class PatientFactory:
    @staticmethod
    def create(record: dict) -> BasePatient:
        # If the condition is one of the common healthcare conditions,
        # return a more detailed patient object.
        if record['condition'] in {'Diabetes', 'Hypertension'}:
            return HealthcarePatient(record['patient_id'], record['age'], record['condition'], record['department'])
        # Otherwise, return the base patient object.
        return BasePatient(record['patient_id'], record['age'])


# Take the first row from the healthcare dataset as an example input.
row = df.iloc[0].to_dict()

# Ask the factory to create the best-matching object.
created = PatientFactory.create(row)

# Show the type of object created.
print(type(created).__name__)

# Show how the object describes itself.
print(created.describe())

## 5. Singleton Pattern

This section shows how one shared configuration object can be reused everywhere.

- `PipelineConfig()` creates only one instance.
- Every later call returns the same object.
- This is helpful for global settings like folder path and encoding.

Why this is useful:
It avoids creating multiple copies of the same configuration and keeps the pipeline consistent.


In [ ]:
# ------------------------------------------------------------------
# Cell 6: Singleton pattern
# ------------------------------------------------------------------
# A singleton ensures that only one shared object is created.
# This is useful for global settings such as output directory and encoding.
class PipelineConfig:
    _instance = None

    def __new__(cls):
        # If no instance exists yet, create one.
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            # Store shared settings on the singleton object.
            cls._instance.output_dir = 'output'
            cls._instance.encoding = 'utf-8'
        # Return the same object every time.
        return cls._instance


# Create two references to the same config object.
config_a = PipelineConfig()
config_b = PipelineConfig()

# Check whether both variables point to the same object.
print('Same object?', config_a is config_b)

# Print the shared output directory.
print('Output dir:', config_a.output_dir)

## 6. Lab 3: Typed DataPipeline Class

This is the final and most practical part of the notebook.

- `load()` reads the raw healthcare data.
- `transform()` adds new useful calculations and categories.
- `export()` writes the results back to a CSV file.
- Type hints make it clear what each method expects and returns.

This is the same pattern used in real data engineering projects: read data, clean/transform it, and save the result.


In [ ]:
# ------------------------------------------------------------------
# Cell 7: Typed DataPipeline lab
# ------------------------------------------------------------------
# This class is the real pipeline example for Day 3.
# It combines: load -> transform -> export.


class TypedDataPipeline:
    # __init__ prepares the file paths for the pipeline.
    def __init__(self, file_path: str, output_dir: str = 'output') -> None:
        # Use the given path if it exists.
        self.file_path = Path(file_path) if Path(file_path).exists() else Path('Day3_OOP_DataPipeline') / file_path
        # Use the given output folder if it exists; otherwise create the Day3 version.
        self.output_dir = Path(output_dir) if Path(output_dir).exists() else Path('Day3_OOP_DataPipeline') / output_dir
        # Make sure the output folder exists before writing files.
        self.output_dir.mkdir(exist_ok=True)

    # load() reads the raw CSV dataset into memory.
    def load(self) -> pd.DataFrame:
        return pd.read_csv(self.file_path)

    # transform() adds new business-friendly columns to the data.
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        transformed = df.copy()
        # Compute average cost per visit for each patient.
        transformed['cost_per_visit'] = transformed['total_cost'] / transformed['visits']
        # Group ages into meaningful categories.
        transformed['age_group'] = pd.cut(
            transformed['age'],
            bins=[0, 30, 45, 60, 100],
            labels=['Young', 'Adult', 'Senior', 'Elderly'],
            right=True,
        )
        return transformed

    # export() saves the transformed data to a CSV file.
    def export(self, df: pd.DataFrame, filename: str) -> None:
        output_file = self.output_dir / filename
        df.to_csv(output_file, index=False)
        print(f'Exported to {output_file}')


# Create the pipeline object.
pipeline = TypedDataPipeline('data/healthcare.csv')

# Step 1: load the raw data.
raw_df = pipeline.load()

# Step 2: transform the data with new columns.
prepared_df = pipeline.transform(raw_df)

# Step 3: export the transformed data to a file.
pipeline.export(prepared_df, 'healthcare_pipeline_output.csv')

# Show the first rows of the transformed table.
prepared_df.head()

## 7. Practice Questions

1. What is the difference between a class and an object?
2. Why is inheritance useful in a data pipeline?
3. Where would you use a factory pattern in analytics code?
4. Why are type hints useful for team projects?
5. What is one real-world benefit of a singleton configuration object?
